## Conexion Power Bi - Python

In [ ]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datetime import datetime

from mysql.connector import errorcode

try:
  cnx = mysql.connector.connect(user='root',
                                password = '1234',
                                database='negocio')
except mysql.connector.Error as err:
  if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
    print("Something is wrong with your user name or password")
  elif err.errno == errorcode.ER_BAD_DB_ERROR:
    print("Database does not exist")
  else:
    print(err)

mycursor = cnx.cursor()
mycursor.execute("SHOW TABLES")

tablas = []  # Lista vacía para almacenar los nombres

for x in mycursor.fetchall():  # fetchall() devuelve una lista de tuplas
    tablas.append(x[0])  # Extrae el primer elemento de cada tupla

print(tablas)  # Imprime la lista de nombres de tablas

# Diccionario para almacenar los DataFrames
dfs = {}

# Cargar cada tabla en un DataFrame
for x in tablas:
    mycursor.execute(f"SELECT * FROM {x}")  # Ejecutar consulta
    columnas = [desc[0] for desc in mycursor.description]  # Obtener nombres de columnas
    datos = mycursor.fetchall()  # Obtener datos
    df = pd.DataFrame(datos, columns=columnas)  # Crear DataFrame
    dfs[x] = df  # Guardar en el diccionario

# Cerrar conexión
mycursor.close()
cnx.close()
#Imprimo los nombres de las tablas 
for x in dfs:
    print(x)

companies_DF = dfs["companies"]
credit_cards_DF = dfs["credit_cards"]
estado_tarjetas_DF = dfs["estado_tarjetas"]
products_DF = dfs["products"]
trans_prod_DF = dfs["trans_prod"]
transaction_DF = dfs["transaction"]
user_DF = dfs["user"]

credit_cards_DF['expiring_date'] = pd.to_datetime(credit_cards_DF['expiring_date'], errors='coerce')

products_DF['price'] = products_DF['price'].str.replace('$', '')
products_DF['price'] = products_DF['price'].astype(float) 

transaction_DF['lat'] = transaction_DF['lat'].astype(float)
transaction_DF['longitude'] = transaction_DF['longitude'].astype(float)
transaction_DF['declined'] = transaction_DF['declined'].astype(int).astype(bool)
transaction_DF["anyo"] = transaction_DF["timestamp"].dt.year
transaction_DF["mes"] = transaction_DF["timestamp"].dt.month
transaction_DF['dia'] = transaction_DF["timestamp"].dt.day_name()

#Funcion para convertir una fecha de nacimiento en Edad 
def edad(fecha_nacimiento):
    hoy = datetime.today()
    return hoy.year - fecha_nacimiento.year - ((hoy.month, hoy.day) < (fecha_nacimiento.month, fecha_nacimiento.day))

user_DF['birth_date'] = pd.to_datetime(user_DF['birth_date'], errors='coerce')
user_DF['Edad'] = user_DF['birth_date'].apply(edad)
user_DF["nombre_completo"] = user_DF["name"] + " " + user_DF["surname"]

In [1]:
## Graficos

In [ ]:
#Grafico 1

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

#Establecer el estilo del grafico y el tamaño
sns.set_theme(style="whitegrid") 

plt.figure(figsize=(13, 10))

# Titulos y nombre de eje Y

plt.title("Porcentaje de precios por productos vendidos", fontsize=22, fontweight="bold")

plt.ylabel("Porcentaje por precio", fontsize=14)
plt.xlabel("Precio", fontsize=14)


#Crear grafico, con eje x, 5 rangos de edad y agrupando por porcentaje
sns.histplot(dataset, x="price", bins=8, stat='percent') 
plt.show()

In [ ]:
#Grafico 2
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

dataset["rango_edad"] = pd.cut(dataset["Edad"], bins=[24,30,36,42,48], labels=["24-30", "30-36", "36-42", "42-48"])

plt.figure(figsize=(13, 10))
sns.boxplot(x="rango_edad", y="amount", data=dataset) #Creo el grafico (boxplot) con los rangos en el eje x y amount en y
#Etiqueto ambos ejes y asigno titulo
plt.xlabel("Grupo de Edad")
plt.ylabel("Transacciones")
plt.title("Distribución de Gasto por Grupo de Edad", fontsize=22)
#Muestro el grafico
plt.show()


#Al trabajar con medias, este grafico es muy diferete de la version de python donde se unen los DF y se cambia el valor de amount

In [ ]:
#Grafico 3
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 11))

# Datos ajustados para el gráfico
data = dataset.groupby("country")["company_name"].nunique().reset_index(name="num_empresas").sort_values('num_empresas', ascending=False)

# Crear el gráfico de barras
sns.barplot(
    data=data,
    x="country",
    y="num_empresas",
    order=data['country']
)

# Mejoras de visualización
plt.xticks(rotation=90, fontsize=10)  # Rotar etiquetas y aumentar tamaño
plt.title("Cantidad de empresas por país", fontsize=14)  # Título del gráfico
plt.xlabel("País", fontsize=12)  # Etiqueta del eje X
plt.ylabel("Número de empresas", fontsize=12)  # Etiqueta del eje Y

# Mostrar el gráfico
plt.show()


In [ ]:
#Grafico 4 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Supongamos que tienes un DataFrame llamado df_ventas
data = dataset.groupby("dia").sum(numeric_only=True).reset_index()

# Ordenar los días de la semana
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Crear el gráfico de barras
plt.figure(figsize=(13, 10))
sns.barplot(data=data, x='dia', y='amount', order=order)

# Personalizar el gráfico
plt.title('Ventas por Día de la Semana', fontsize=20)
plt.xlabel('Día de la Semana')
plt.ylabel('Monto de Ventas')

# Mostrar el gráfico
plt.show()

In [ ]:
#Graafico 5

import seaborn as sns
import matplotlib.pyplot as plt

# Agrupar y contar las ventas por país y producto
top_products = dataset.groupby(["country", "product_name"]).size().reset_index(name="count")

# Filtrar solo los 10 productos más vendidos por país
top_products = top_products.groupby("country").apply(lambda x: x.nlargest(10, "count")).reset_index(drop=True)

# Crear el gráfico
plt.figure(figsize=(15, 8))
sns.barplot(data=top_products, x="country", y="count", hue="product_name")
plt.title("Top 10 Productos más vendidos por País", fontsize=25)
plt.xticks(rotation=45)
plt.legend(title="Producto", bbox_to_anchor=(1, 1), loc='best')
plt.show()


In [ ]:
#Grafico 6

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.set(style="whitegrid")

# Crear el gráfico de barras sin asignar a 'ax'
sns.barplot(data=dataset, x="dia", y="amount", hue="declined", dodge=False, ci=False, estimator=np.sum)

# Añadir títulos y etiquetas
plt.title("Total de Transacciones por Día", fontsize=20)
plt.xlabel("Día de la Semana")
plt.ylabel("Total de Transacciones")
plt.legend(title="Declinación")

# Mostrar el gráfico
plt.show()

In [ ]:
#Grafico 7 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.pairplot(dataset,  x_vars=["amount", "Edad", "price"], y_vars=["amount", "Edad", "price"], hue="anyo", palette="tab10")
plt.suptitle("Relación entre Amount, Edad y Price por Año", fontsize=16)

plt.subplots_adjust(top=0.95)  # Ajuste para no superponer el título

plt.show()

In [ ]:
#Grafico 8

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
df_numericas = dataset.select_dtypes(include ='number')#.style.background_gradient('coolwarm')
corr_matrix = df_numericas.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.1, vmin=-1, vmax=1)
plt.title("Matriz de Correlación")
plt.show()

In [ ]:
#Grafico 9 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Crear el jointplot 
g = sns.jointplot(data=dataset, x="Edad", y="amount", hue="country", palette="Set2")

g.fig.set_size_inches(9, 8)

# Modificar las etiquetas
g.set_axis_labels("Edad del Usuario", "Monto de la Venta", fontsize=12)

# Mover la leyenda
g.ax_joint.legend(title="País del Usuario", loc='upper left', bbox_to_anchor=(1, 1.3), fontsize=10)

# Ajustar márgenes
#g.fig.subplots_adjust(top=1, right=0.9, left=0.2, bottom=0.2)

# Mostrar el gráfico
plt.show()



In [ ]:
#Grafico 10
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.figure(figsize=(9, 7))


sns.violinplot(x='country', y='amount', data=dataset, inner=None, color=".8")
sns.barplot(x='country', y='amount', data=dataset, color='lightblue', alpha=0.6)
plt.axhline(y=dataset['amount'].mean(), color='r', linestyle='--')

plt.title('Distribución de Amount por Pais del usuario')
plt.xlabel('Pais de Usuario')
plt.ylabel('Monto de transaccion')

plt.show()

In [ ]:
#Grafico 11
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.figure(figsize=(9, 7))


sns.violinplot(x='country', y='amount', data=dataset, inner=None, color=".8")
sns.barplot(x='country', y='amount', data=dataset, color='lightblue', alpha=0.6)
plt.axhline(y=dataset['amount'].mean(), color='r', linestyle='--')

plt.title('Distribución de Amount por Pais del usuario')
plt.xlabel('Pais de Usuario')
plt.ylabel('Monto de transaccion')

plt.show()